# E06 - CHẤT LƯỢNG DỮ LIỆU: GẦN TRÙNG, RÒ RỈ & BẮT 4–5★ KHIẾU NẠI NGẦM (NGƯỜI 1)

**Vấn đề:**
1. NB01 mới lọc trùng khớp 100% `sentence_clean`, và áp 2 luật khác nhau giữa train/dev. Các câu chỉ khác dấu câu hoặc emoji vẫn có thể lọt qua ranh giới train–dev/test.
2. Ngoài thực tế TMĐT, bộ lọc truyền thống chỉ nhìn `n_star <= 2` sẽ bỏ sót các đánh giá **4–5 sao nhưng nội dung chê thật** (*Ý tưởng 1*).

In [1]:
import os, sys, glob, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

def _find_root():
    cwd = os.getcwd()
    cand = os.path.abspath('..') if os.path.basename(cwd) in ('notebooks', 'experiments') else cwd
    if os.path.exists(os.path.join(cand, 'src', 'modeling.py')):
        return cand, cand
    if IN_COLAB:
        for base in [cwd, '/content']:
            if os.path.exists(os.path.join(base, 'src', 'modeling.py')):
                return base, base
        for z in glob.glob('/content/*src_and_data*.zip'):
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall('/content')
            if os.path.exists('/content/src/modeling.py'):
                return '/content', '/content'
    if IN_KAGGLE:
        work = '/kaggle/working'
        if os.path.exists(os.path.join(work, 'src', 'modeling.py')):
            return work, work
        for p in glob.glob('/kaggle/input/**/src/modeling.py', recursive=True):
            return os.path.dirname(os.path.dirname(p)), work
        for z in glob.glob('/kaggle/input/**/*src_and_data*.zip', recursive=True):
            dest = os.path.join(work, 'project')
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(dest)
            if os.path.exists(os.path.join(dest, 'src', 'modeling.py')):
                return dest, dest
    raise FileNotFoundError('Khong tim thay src/modeling.py! Hay chay trong repo hoac upload src_and_data.zip.')

ROOT, WORK_DIR = _find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
FIG_DIR = os.path.join(WORK_DIR, 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
os.environ['EXP_LOG_PATH'] = os.path.join(WORK_DIR, 'reports', 'experiments_log.csv')
print(f'Env: colab={IN_COLAB}, kaggle={IN_KAGGLE} | ROOT={ROOT}')

Env: colab=False, kaggle=False | ROOT=D:\E-commerce


In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, precision_score, recall_score
from src.labels import ASPECT_GROUPS
from src.exp_log import log_experiment

df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
df_test = pd.read_csv(os.path.join(PROC_DIR, 'test_clean.csv'))
print('Shapes -> train:', df_train.shape, '| dev:', df_dev.shape, '| test:', df_test.shape)

Shapes -> train: (7776, 16) | dev: (1110, 16) | test: (2224, 16)


## 1. KIỂM TRA TRÙNG LẶP NỘI BỘ KHI BỎ DẤU CÂU

In [3]:
def norm_loose(s):
    return re.sub(r'[^\w\s]', ' ', str(s).lower()).split()

for name, df in [('train', df_train), ('dev', df_dev)]:
    loose = df['sentence_clean'].fillna('').apply(lambda s: ' '.join(norm_loose(s)))
    tmp = df.assign(loose=loose)
    grp_cnt = tmp.groupby('loose')['sentiment'].agg(['size', 'nunique'])
    dups = grp_cnt[grp_cnt['size'] > 1]
    conflicts = dups[dups['nunique'] > 1]
    print(f'{name}: nhom trung sau bo dau cau = {len(dups)}, trong do mau thuan nhan sentiment = {len(conflicts)}')

train: nhom trung sau bo dau cau = 6, trong do mau thuan nhan sentiment = 0
dev: nhom trung sau bo dau cau = 0, trong do mau thuan nhan sentiment = 0


## 2. ĐO CÂU GẦN TRÙNG (NEAR-DUPLICATE) GIỮA TRAIN – DEV VÀ TRAIN – TEST

In [4]:
# Dung TF-IDF char_wb (3, 5) de kiem toan (audit) cau giong nhau giua train voi dev/test
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2)
M_tr = char_vec.fit_transform(df_train['sentence_clean'].fillna(''))
M_dev = char_vec.transform(df_dev['sentence_clean'].fillna(''))
M_te = char_vec.transform(df_test['sentence_clean'].fillna(''))

sim_dev = (M_dev @ M_tr.T).toarray()
sim_te = (M_te @ M_tr.T).toarray()
dev_max_sim = sim_dev.max(axis=1)
dev_best_idx = sim_dev.argmax(axis=1)
te_max_sim = sim_te.max(axis=1)

# Luu y: chi do tan suat tren test de audit, tuyet doi KHONG dung sim_te de loc train
for th in [0.80, 0.90, 0.95]:
    print(f'Cosine >= {th:.2f} -> dev: {(dev_max_sim >= th).sum()}/{len(df_dev)} | test (audit only): {(te_max_sim >= th).sum()}/{len(df_test)}')

pairs = []
for i in np.argsort(-dev_max_sim)[:12]:
    j = int(dev_best_idx[i])
    pairs.append({
        'cos': round(float(dev_max_sim[i]), 3),
        'dev_sent': int(df_dev.iloc[i]['sentiment']),
        'tr_sent': int(df_train.iloc[j]['sentiment']),
        'dev_text': df_dev.iloc[i]['sentence_clean'][:75],
        'train_text': df_train.iloc[j]['sentence_clean'][:75],
    })
pd.DataFrame(pairs)

Cosine >= 0.80 -> dev: 10/1110 | test (audit only): 29/2224
Cosine >= 0.90 -> dev: 5/1110 | test (audit only): 18/2224
Cosine >= 0.95 -> dev: 1/1110 | test (audit only): 14/2224


,cos,dev_sent,tr_sent,dev_text,train_text
0,0.986,2,2,Sài rất tốt chụp ngoại trời siêu sắc nét đẹp c...,"sản phẩm tốt, giá cả hợp lí ....................."
1,0.941,0,2,"Zo google hay bi không phản hồi , đứng máy ,mớ...","sản phẩm tốt, giá cả hợp lí ....................."
2,0.939,1,2,Cảm ứng còn đơ..để xài 1 2 ngày xem sao.chơi g...,Mình sài gần được 1 tháng rồi. Thấy rất ok.......
3,0.917,2,2,Mình có những đánh giá như sau: - Màn hình đẹp...,Mình có những đánh giá như sau: - Màn hình đẹp...
4,0.915,1,1,Sản phẩm dùng cho nghe gọi ok. Loa lớn. Màu sắ...,Sản phẩm dùng cho nghe gọi ok. Loa lớn. Màu sắ...
5,0.847,0,0,"Hình như chỉ có 1 bên loa nghe được thôi, mình...","Hình như chỉ có 1 bên loa nghe được thôi, mình..."
6,0.835,2,2,Chiến game maxx ngon. Nhân viên tận tình. Anh ...,Máy ngon loa to . Chơi game liên quân khá ổn f...
7,0.828,2,2,"Máy ngon, đẹp, pin trâu. Mọi thứ đều ok trừ vi...","Máy ngon, đẹp, pin trâu. Mọi thứ đều ok trừ vi..."
8,0.827,2,2,"Máy ok. Chơi game tốt, camera lấy nét cực kì. ...","Máy ok. Chơi game tốt, camera 64mp lấy nét cực..."
9,0.810,2,2,Tuyệt vời quá luôn ý. Đã là 1 tín đồ mê dòng đ...,mua được 3 ngày vẫn chưa thây vấn đề gì.sài rấ...


## 3. ĐO TÁC ĐỘNG CỦA RÒ RỈ GẦN TRÙNG LÊN DEV MACRO F1

In [5]:
TOKEN_PAT = r'(?u)\b\w+\b|\[\w+\]'
def eval_train_variant(df_tr_sub, run_name, desc):
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)),
    ]).fit(df_tr_sub['sentence_seg'].fillna(''), df_tr_sub['sentiment'])
    pred = pipe.predict(df_dev['sentence_seg'].fillna(''))
    f1_all = f1_score(df_dev['sentiment'], pred, average='macro')
    mask_nd = dev_max_sim >= 0.90
    f1_clean = f1_score(df_dev.loc[~mask_nd, 'sentiment'], pred[~mask_nd], average='macro')
    f1_nd = f1_score(df_dev.loc[mask_nd, 'sentiment'], pred[mask_nd], average='macro') if mask_nd.sum() >= 5 else np.nan
    log_experiment('E06', run_name, desc, {'n_train': len(df_tr_sub)}, dev_f1=f1_all,
                   notes=f'dev_clean_slice={f1_clean:.4f}, dev_neardup_slice={f1_nd:.4f} (n={int(mask_nd.sum())})')
    return {'run': run_name, 'n_train': len(df_tr_sub), 'dev_f1': round(f1_all, 4),
            'clean_slice_f1': round(f1_clean, 4), 'neardup_slice_f1': None if np.isnan(f1_nd) else round(float(f1_nd), 4)}, pipe

# Bien the (a) hien tai; (b) chi loai near-dup >= 0.90 theo dev (KHONG nhin test); (c) bo them nhom mau thuan noi bo train
bad_tr_idx = set(np.where(sim_dev.max(axis=0) >= 0.90)[0])
df_tr_b = df_train.drop(index=list(bad_tr_idx)).reset_index(drop=True)

loose_tr = df_tr_b['sentence_clean'].fillna('').apply(lambda s: ' '.join(norm_loose(s)))
grp_n = df_tr_b.assign(loose=loose_tr).groupby('loose')['sentiment'].transform('nunique')
df_tr_c = df_tr_b[grp_n == 1].reset_index(drop=True)

res_rows = []
r_a, base_pipe = eval_train_variant(df_train, 'a_current_train', 'Baseline train dedup')
r_b, _ = eval_train_variant(df_tr_b, 'b_drop_neardup_090', 'Drop train near-dup >= 0.90 vs dev only (no test leakage)')
r_c, _ = eval_train_variant(df_tr_c, 'c_consistent_dedup', 'Drop near-dup (dev-only) + internal conflicting groups')
pd.DataFrame([r_a, r_b, r_c])

,run,n_train,dev_f1,clean_slice_f1,neardup_slice_f1
0,a_current_train,7776,0.7478,0.7461,1.0000
1,b_drop_neardup_090,7744,0.7449,0.7446,0.8222
2,c_consistent_dedup,7744,0.7449,0.7446,0.8222


## 4. Ý TƯỞNG 1: PHÁT HIỆN 4–5★ KHIẾU NẠI NGẦM (`SILENT_COMPLAINT`)

In [6]:
# Lay lat cat dev danh gia 4-5 sao nhung noi dung Tieeu cuc (sentiment == 0)
mask_high = df_dev['n_star'].values >= 4
y_true_silent = (df_dev.loc[mask_high, 'sentiment'].values == 0).astype(int)
proba_high = base_pipe.predict_proba(df_dev.loc[mask_high, 'sentence_seg'].fillna(''))
pred_high = base_pipe.predict(df_dev.loc[mask_high, 'sentence_seg'].fillna(''))

print(f'So ca 4-5 sao tren dev: {int(mask_high.sum())} | Trong do noi dung Tieu cuc (bi loc sao bo sot): {int(y_true_silent.sum())} ({y_true_silent.mean():.2%})')

th_rows = []
for th_name, flag in [('argmax_neg', pred_high == 0)] + [(f'P(neg)>={t:.1f}', proba_high[:, 0] >= t) for t in [0.4, 0.5, 0.6, 0.7]]:
    p = precision_score(y_true_silent, flag, zero_division=0)
    r = recall_score(y_true_silent, flag, zero_division=0)
    f1_s = f1_score(y_true_silent, flag, zero_division=0)
    th_rows.append({'rule': th_name, 'flagged': int(flag.sum()), 'tp_caught': int((flag & (y_true_silent == 1)).sum()),
                    'false_alarm': int((flag & (y_true_silent == 0)).sum()),
                    'precision': round(p, 4), 'recall': round(r, 4), 'f1': round(f1_s, 4)})
    if th_name in ('argmax_neg', 'P(neg)>=0.5'):
        log_experiment('E06', f'silent_complaint_{th_name}', 'Detect 4-5* silent complaints (text-only)',
                       {'rule': th_name}, dev_f1=f1_s, notes=f'P={p:.4f}, R={r:.4f}, caught={int((flag & (y_true_silent == 1)).sum())}/{int(y_true_silent.sum())}')

df_silent = pd.DataFrame(th_rows)
df_silent

So ca 4-5 sao tren dev: 701 | Trong do noi dung Tieu cuc (bi loc sao bo sot): 54 (7.70%)


,rule,flagged,tp_caught,false_alarm,precision,recall,f1
0,argmax_neg,61,30,31,0.4918,0.5556,0.5217
1,P(neg)>=0.4,63,30,33,0.4762,0.5556,0.5128
2,P(neg)>=0.5,42,30,12,0.7143,0.5556,0.6250
3,P(neg)>=0.6,25,23,2,0.9200,0.4259,0.5823
4,P(neg)>=0.7,12,11,1,0.9167,0.2037,0.3333


In [7]:
# Xem phan bo khia canh bi che trong nhom 4-5* khieu nai ngam va in vi du
silent_cases = df_dev[mask_high & (df_dev['sentiment'] == 0)].copy()
silent_cases['p_neg'] = proba_high[y_true_silent == 1, 0].round(3)
asp_cols = [f'asp_{i}' for i in range(5)]
print('Tan suat khia canh trong 4-5* khieu nai ngam:')
for i, name in enumerate(ASPECT_GROUPS):
    print(f'  {name}: {int(silent_cases[f"asp_{i}"].sum())}/{len(silent_cases)}')
silent_cases[['n_star', 'p_neg', 'sentence_clean']].head(8)

Tan suat khia canh trong 4-5* khieu nai ngam:
  PERF_FEAT: 42/54
  SCREEN_CAM: 13/54
  BATTERY: 20/54
  PRICE_SER: 12/54
  DESIGN_GEN: 28/54


,n_star,p_neg,sentence_clean
36,4,0.390,"Mua được 10 ngày , đánh giá chung máy ok , pin..."
61,5,0.280,Mới mua! Mọi thứ tốt ( trừ camera) Camera siêu...
84,5,0.715,Mua được hơn tuần. Dùng được tầm 7-9h . tối om...
99,5,0.267,Máy tốt nhưng em hơi thắc mắc tại sai giác cắn...
130,4,0.679,Chơi Liên quân khoảng 2 Tiếng la bi out Ra phả...
146,4,0.172,Theo mình nó cũng có vài lỗi là chip khá yếu c...
189,4,0.589,"Máy nóng kinh, cầm dí vào mặt cảm giác như chu..."
286,4,0.754,Điện thoại em mới mua còn bảo hành nhưng xài h...


## 5. KẾT LUẬN & HƯỚNG GIẢI QUYẾT ĐỀ XUẤT

- **Trùng lặp & rò rỉ (Near-duplicate):**
  - Trong `train` có `6` nhóm trùng sau khi bỏ dấu câu nhưng **0 nhóm mâu thuẫn nhãn**. Tỷ lệ câu gần trùng (`cosine >= 0.90`) giữa `train` và `dev`/`test` rất nhỏ (`5/1110` câu dev; audit trên `test` là `18/2224` — chỉ báo cáo kiểm toán, **tuyệt đối không dùng `test` để lọc `train`**).
  - Do lát cắt `dev` gần trùng rất nhỏ (`n = 5` câu), sau khi loại `32` câu train gần trùng theo `dev` (`7776 → 7744`), điểm `dev_neardup_slice` giảm từ `1.0000 → 0.8222` và `dev_clean_slice` giữ quanh `0.7461 → 0.7446`. Hướng đúng là thống nhất quy tắc dedup khi chia tập gốc (`NB01`), không đụng vào `test`.
- **Ý tưởng 1 — Bắt 4–5★ khiếu nại ngầm (`SILENT_COMPLAINT`):**
  - Có `54/701` (`7.70%`) đánh giá 4–5★ trên dev mang nội dung Tiêu cực (bộ lọc theo sao truyền thống bỏ lọt hoàn toàn).
  - Mô hình chỉ đọc văn bản ở ngưỡng mặc định (`argmax`) bắt được `30/54` ca (Recall `55.56%`) nhưng báo nhầm `31` ca (Precision `49.18%`).
  - **Hướng giải quyết (đã đưa vào `NB06`):** giữ nguyên nguyên tắc không dùng `n_star` làm feature, nhưng ở tầng hậu xử lý áp ngưỡng tin cậy **`P(neg) >= 0.50`** (`Khieu_Nai_Ngam_4_5_Sao`, vẫn bắt `30/54` ca mà giảm báo nhầm còn `12` ca → Precision tăng lên **`71.43%`**, F1 = `0.6250`; hoặc `P(neg) >= 0.60` cho Precision **`92.00%`**), đồng thời định tuyến theo khía cạnh (`PERF_FEAT` 42/54 ca, `BATTERY` 20/54 ca) đẩy vào `needs_review`.